## Analysis


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr, linregress
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

### Load Data

In [4]:
df = pd.read_csv("../data/processed/merged_dataset.csv")

print(df.shape)

df.head()

(100, 1096)


,file_id,filepath,age,sex_x,brain_health_score,total_cognition_score,fluid_cognition_score,crystallized_cognition_score,lhl_1,lhl_2,...,BT2.h.7,SGOT2.h.8,SGPT,Family _History,Medical_History,Barthel_scale,MRS_scale,NIHSS_scale,AHI_1_B,subject_id
0,SN1,sleep_data/SN1.edf,65.0,1,-0.27464,-0.01407,-0.10628,0.08692,0.022346,-0.994057,...,1.14,29.8,25.7,NaN,NaN,100.0,0.0,0.0,50.8,SN1
1,SN2,sleep_data/SN2.edf,60.0,1,-0.36531,0.06125,-0.09295,0.30379,0.152037,-0.997863,...,NaN,30.1,26.8,NaN,NaN,100.0,0.0,0.0,6.5,SN2
2,SN3,sleep_data/SN3.edf,49.0,1,0.24275,0.49118,0.45108,0.23969,0.210161,-0.920484,...,NaN,NaN,NaN,NaN,NaN,100.0,0.0,0.0,41.1,SN3
3,SN4,sleep_data/SN4.edf,47.0,1,0.16931,0.49646,0.48187,0.24427,0.123935,-0.969290,...,0.78,23,25.0,1,NaN,100.0,0.0,0.0,39.1,SN4
4,SN5,sleep_data/SN5.edf,39.0,1,0.00631,0.31138,0.31247,-0.02889,0.179273,-0.829033,...,0.70,30,40.0,NaN,NaN,100.0,0.0,0.0,1.9,SN5


In [8]:
x = df["AHI_1_B"].values

In [ ]:
df.columns.tolist()

In [10]:
targets = [
    ("brain_health_score", "Brain Health Score"),
    ("total_cognition_score", "Total Cognition Score"),
    ("fluid_cognition_score", "Fluid Cognition Score"),
    ("crystallized_cognition_score", "Crystallized Cognition Score")
]

In [11]:
for column, title_name in targets:

    y = df[column].values

    # Pearson Correlation
    pearson_r, pearson_p = pearsonr(x, y)

    # Spearman Correlation
    spearman_rho, spearman_p = spearmanr(x, y)

    print("\n" + "="*70)
    print(title_name)
    print("="*70)

    print(f"Pearson r = {pearson_r:.4f}")
    print(f"Pearson p = {pearson_p:.6f}")

    print(f"Spearman rho = {spearman_rho:.4f}")
    print(f"Spearman p   = {spearman_p:.6f}")


Brain Health Score
Pearson r = -0.1791
Pearson p = 0.074629
Spearman rho = -0.1591
Spearman p   = 0.113839

Total Cognition Score
Pearson r = -0.2364
Pearson p = 0.017870
Spearman rho = -0.2057
Spearman p   = 0.040093

Fluid Cognition Score
Pearson r = -0.2144
Pearson p = 0.032185
Spearman rho = -0.1722
Spearman p   = 0.086620

Crystallized Cognition Score
Pearson r = 0.0223
Pearson p = 0.825459
Spearman rho = 0.0571
Spearman p   = 0.572274


In [ ]:
for column, title_name in targets:

    # Target variable
    y = df[column].values

    # Correlation Analysis
    pearson_r, pearson_p = pearsonr(x, y)
    spearman_rho, spearman_p = spearmanr(x, y)

    print("\n" + "=" * 70)
    print(title_name)
    print("=" * 70)

    print(f"Pearson Correlation : r = {pearson_r:.4f}, p = {pearson_p:.6f}")
    print(f"Spearman Correlation: rho = {spearman_rho:.4f}, p = {spearman_p:.6f}")

    
    # Linear Regression (OLS)
    linear_result = linregress(x, y)

    slope = linear_result.slope
    intercept = linear_result.intercept

    print("\nLinear Regression (OLS)")
    print(f"Slope (w)     = {slope:.6f}")
    print(f"Intercept (b) = {intercept:.6f}")


    # Polynomial Regression (Degree 2)
    X = x.reshape(-1, 1)

    poly = PolynomialFeatures(degree=2)
    X_poly = poly.fit_transform(X)

    poly_model = LinearRegression()
    poly_model.fit(X_poly, y)

    print("\nPolynomial Regression")

    print(
        f"y = {poly_model.coef_[2]:.6f}x² "
        f"+ {poly_model.coef_[1]:.6f}x "
        f"+ {poly_model.intercept_:.6f}"
    )

  
    # Generate Smooth Curves
    x_plot = np.linspace(x.min(), x.max(), 500)

    y_linear = slope * x_plot + intercept

    y_poly = poly_model.predict(
        poly.transform(
            x_plot.reshape(-1, 1)
        )
    )

    # ---------------------------------
    # Plot
    # ---------------------------------
    plt.figure(figsize=(10, 6))

    plt.scatter(
        x,
        y,
        s=60,
        alpha=0.7,
        label="Subjects"
    )

    plt.plot(
        x_plot,
        y_linear,
        linewidth=2,
        label="Linear Fit (OLS)"
    )

    plt.plot(
        x_plot,
        y_poly,
        "--",
        linewidth=2,
        label="Polynomial Fit (Degree 2)"
    )

    plt.xlabel("AHI_1_B", fontsize=12)
    plt.ylabel(title_name, fontsize=12)

    plt.title(
        f"{title_name} vs AHI_1_B\n"
        f"Pearson r = {pearson_r:.3f} (p = {pearson_p:.4f})"
    )

    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    # Save plot
    file_name = f"../outputs/plots/{column}_analysis.png"

    plt.savefig(
        file_name,
        dpi=300,
        bbox_inches="tight"
    )

    print(f"\nSaved Plot: {file_name}")

    plt.show()